# Proyecto Final: comparación de clasificadores

**Asignatura:** Aprendizaje Computacional  
**Programa:** Ingeniería de Sistemas  
**Grupo:** Robin Alexander Montoya, Tomas Correa y Samuel Arango

## Dataset y problema de clasificación

El problema consiste en clasificar si un cliente suscribe (`yes`) o no (`no`) un depósito a plazo tras una campaña de marketing telefónico. Se trata de una clasificación binaria. El CSV contiene **11.162 registros**, **16 variables de entrada** y una variable objetivo: `deposit`, que se asignará a `y`.

### Enlace y procedencia

El dataset se obtuvo de [Bank Marketing Dataset en Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset), publicado por Janio Martinez Bachmann. La descripción de Kaggle cita como fuente a S. Moro, P. Cortez y P. Rita, “A Data-Driven Approach to Predict the Success of Bank Telemarketing”, *Decision Support Systems*, 62:22–31, 2014. La copia distribuida con este proyecto es `data/bank.csv`. El notebook la carga con una ruta relativa; si no está disponible, descarga el dataset público con `kagglehub`, sin requerir credenciales para la copia incluida.

### Variables disponibles

| Variable | Descripción |
|---|---|
| `age` | Edad del cliente. |
| `job` | Ocupación. |
| `marital` | Estado civil. |
| `education` | Nivel educativo. |
| `default` | Si tiene crédito en incumplimiento. |
| `balance` | Balance medio anual. |
| `housing` | Si tiene préstamo hipotecario. |
| `loan` | Si tiene préstamo personal. |
| `contact` | Tipo de medio de contacto. |
| `day` | Día del mes del último contacto. |
| `month` | Mes del último contacto. |
| `duration` | Duración del último contacto, en segundos. |
| `campaign` | Número de contactos durante esta campaña. |
| `pdays` | Días desde un contacto en una campaña anterior; `-1` indica que no fue contactado previamente. |
| `previous` | Número de contactos antes de esta campaña. |
| `poutcome` | Resultado de la campaña anterior. |
| `deposit` | **Objetivo (`y`):** si suscribió un depósito a plazo (`yes`/`no`). |

La distribución observada en el CSV es `no`: 5.873 (52,62 %) y `yes`: 5.289 (47,38 %).

## Modelos e hiperparámetros

Se comparan tres algoritmos de clasificación con cinco valores candidatos para cada hiperparámetro:

| Modelo | Hiperparámetro | Valores evaluados | Justificación |
|---|---|---|---|
| Regresión logística | `C` | `0.01`, `0.1`, `1`, `10`, `100` | Modelo lineal interpretable; `C` controla la regularización. |
| Redes Neuronales | `hidden_layer_sizes` | `(10,)`, `(50,)`, `(100,)`, `(50, 50)`, `(100, 100)` | Modelo no lineal; permite explorar la capacidad de representación. |
| K vecinos más cercanos (KNN) | `n_neighbors` | `1`, `3`, `5`, `9`, `15` | Método basado en distancias; permite analizar la importancia del escalado. |

## Carga de datos y separación entre X e y

Se busca primero el CSV incluido en el proyecto, tanto si Jupyter se inicia desde la carpeta raíz como desde `notebooks/`. Solo si no se encuentra se usa la descarga pública de Kaggle.

In [40]:
from pathlib import Path

import pandas as pd

project_roots = (Path.cwd(), Path.cwd().parent)
data_path = next(
    (root / "data" / "bank.csv" for root in project_roots
     if (root / "data" / "bank.csv").is_file()),
    None,
)

if data_path is None:
    import kagglehub

    downloaded_dir = Path(
        kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")
    )
    downloaded_files = list(downloaded_dir.rglob("bank.csv"))
    if len(downloaded_files) != 1:
        raise FileNotFoundError(
            "No se encontró un único archivo bank.csv en el dataset descargado. "
            f"Archivos encontrados: {downloaded_files}"
        )
    data_path = downloaded_files[0]

df = pd.read_csv(data_path)
target_column = "deposit"
if target_column not in df.columns:
    raise ValueError(f"El CSV no contiene la columna objetivo {target_column!r}.")

X = df.drop(columns=target_column)
y = df[target_column]

print(f"Archivo cargado: {data_path}")
print(f"Registros: {len(df):,}; variables de entrada: {X.shape[1]}")
print(f"Clases de y: {y.nunique()}")
display(df.head())

Archivo cargado: c:\Users\Robin Montoya\OneDrive - INSTITUTO TECNOLOGICO METROPOLITANO - ITM\Documents\ComparacionDeClasificadores\data\bank.csv
Registros: 11,162; variables de entrada: 16
Clases de y: 2


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [41]:
cantidad = (df['poutcome'] == 'unknown').sum()
print(cantidad)

8326


se borra poutcome ya que tiene muchos valores desconocidos y no aporta información relevante para el modelo ya que la mayoría de los registros tienen este valor. Se decide eliminar esta variable del dataset para evitar que afecte negativamente el rendimiento de los modelos de clasificación.

In [42]:
df = df.drop(columns=['poutcome'])

se borra el tiempo de contacto ya que no aporta información relevante para el modelo y puede introducir ruido en los datos. Además, la duración del contacto no es una variable que se pueda controlar o predecir, por lo que se considera irrelevante para la tarea de clasificación.

In [43]:
df = df.drop(columns=['duration'])

In [44]:
distribution = y.value_counts().rename_axis("Clase").to_frame("Registros")
distribution["Porcentaje"] = (distribution["Registros"] / len(y) * 100).round(2)
display(distribution)

,Registros,Porcentaje
Clase,,
no,5873,52.62
yes,5289,47.38


cambio de las varibles categóricas a numéricas, y la separación de las variables de entrada `X` y la variable objetivo `y`. Se realiza un preprocesamiento básico para convertir las variables categóricas en variables dummy (one-hot encoding) y se asegura que la variable objetivo sea binaria.

```python

In [45]:
df['default'] = df['default'].map({'yes': 1, 'no': 0})
df['housing'] = df['housing'].map({'yes': 1, 'no': 0})
df['loan'] = df['loan'].map({'yes': 1, 'no': 0})
df['deposit'] = df['deposit'].map({'yes': 1, 'no': 0})
df['marital'] = df['marital'].map({'married': 1, 'single': 0, 'divorced': 2})



In [46]:
moda_contact = df[df['contact'] != 'desconocido']['contact'].mode()[0]
print(f"Moda de la variable 'contact' (excluyendo 'desconocido'): {moda_contact}")


Moda de la variable 'contact' (excluyendo 'desconocido'): cellular


 Reemplazar 'desconocido' por esa moda

In [47]:
df['contact'] = df['contact'].replace('desconocido', moda_contact)

## Exploración y calidad de los datos

Se revisan los valores faltantes y los tipos de datos de las variables antes de continuar con el modelado.

In [48]:
df.isnull().sum()


age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
campaign     0
pdays        0
previous     0
deposit      0
dtype: int64

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        11162 non-null  int64
 1   job        11162 non-null  str  
 2   marital    11162 non-null  int64
 3   education  11162 non-null  str  
 4   default    11162 non-null  int64
 5   balance    11162 non-null  int64
 6   housing    11162 non-null  int64
 7   loan       11162 non-null  int64
 8   contact    11162 non-null  str  
 9   day        11162 non-null  int64
 10  month      11162 non-null  str  
 11  campaign   11162 non-null  int64
 12  pdays      11162 non-null  int64
 13  previous   11162 non-null  int64
 14  deposit    11162 non-null  int64
dtypes: int64(11), str(4)
memory usage: 1.3 MB


## División en entrenamiento y prueba

Se divide el conjunto de datos en entrenamiento (80 %) y prueba (20 %). La división es estratificada para conservar la proporción de las dos clases (`yes` y `no`) en ambos conjuntos y utiliza una semilla fija para que el resultado sea reproducible.

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Entrenamiento (Train): {X_train.shape[0]} filas")
print(f"Prueba (Test): {X_test.shape[0]} filas")

print("\nDistribución de clases en Entrenamiento:")
print(y_train.value_counts())

print("\nDistribución de clases en Prueba:")
print(y_test.value_counts())

Entrenamiento (Train): 8929 filas
Prueba (Test): 2233 filas

Distribución de clases en Entrenamiento:
deposit
no     4698
yes    4231
Name: count, dtype: int64

Distribución de clases en Prueba:
deposit
no     1175
yes    1058
Name: count, dtype: int64
